# Segmentación de clientes con K-Means
**Aprendizaje no supervisado · práctica de 4 celdas**  
**Docente: Ing. Freddy Martínez**

Una tienda ficticia quiere agrupar compradores según su comportamiento y proponer
campañas distintas para cada grupo. **El modelo forma grupos; las campañas son propuestas
humanas posteriores, no recomendaciones aprendidas por K-Means.**

Abre este cuaderno en Google Colab y ejecuta las cuatro celdas en orden.
En la celda 1, carga **clientes_segmentacion.csv**. No necesitas conectar Drive ni utilizar claves.

## ¿Qué contiene el CSV?
Son **60 clientes ficticios**, una fila por cliente, con actividad en los últimos **90 días**
respecto de una misma fecha de corte ficticia. No son datos reales ni resultados de una encuesta.
Los patrones se diseñaron para facilitar la visualización de grupos.

| Columna | Significado |
|---|---|
| `cliente_id` | Identificador ficticio. **No se usa para entrenar**. |
| `compras_90_dias` | Número de compras durante los 90 días. |
| `gasto_90_dias_cop` | Gasto total durante ese periodo, en pesos colombianos, sin separadores de miles. |
| `dias_ultima_compra` | Días desde la compra más reciente hasta el corte: 0 significa que compró ese día. |

No hay una columna con el segmento correcto ni una variable objetivo `y`.
Todos los clientes compraron al menos una vez en ese periodo; por eso los días van de 0 a 89.

## Celda 1. Cargar y revisar los clientes
Selecciona **solo el CSV suministrado**. Se verifican columnas, identificadores y valores numéricos.
Verás la cantidad de clientes y las primeras cinco filas. El original no se modifica.

Si aparece un error de biblioteca no instalada, ejecuta `%pip install pandas numpy matplotlib scikit-learn`
en una celda adicional y vuelve a comenzar. No es necesario si las importaciones funcionan.

In [ ]:
# CELDA 1. CARGAR Y REVISAR
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

archivos = files.upload()  # Selecciona clientes_segmentacion.csv.
if len(archivos) != 1:
    raise ValueError("Selecciona exactamente un archivo CSV.")
nombre, contenido = next(iter(archivos.items()))
if not nombre.lower().endswith(".csv"):
    raise ValueError("Debes seleccionar el archivo CSV, no el cuaderno ni un ZIP.")
clientes = pd.read_csv(io.BytesIO(contenido), encoding="utf-8-sig")

VARIABLES = ["compras_90_dias", "gasto_90_dias_cop", "dias_ultima_compra"]
columnas = ["cliente_id"] + VARIABLES
if not set(columnas).issubset(clientes.columns):
    raise ValueError(f"El CSV debe contener estas columnas: {columnas}")
clientes = clientes[columnas].copy()
if clientes["cliente_id"].isna().any():
    raise ValueError("Hay identificadores vacíos.")
clientes["cliente_id"] = clientes["cliente_id"].astype(str).str.strip()
if clientes["cliente_id"].eq("").any() or clientes["cliente_id"].duplicated().any():
    raise ValueError("Cada cliente debe tener un identificador único y no vacío.")
clientes[VARIABLES] = clientes[VARIABLES].apply(pd.to_numeric, errors="coerce")
if len(clientes) < 3 or not np.isfinite(clientes[VARIABLES].to_numpy()).all():
    raise ValueError("Se requieren al menos 3 clientes y números válidos, sin vacíos.")
if (clientes[VARIABLES[:2]] <= 0).any().any() or not clientes[VARIABLES[2]].between(0, 89).all():
    raise ValueError("Compras y gasto deben ser positivos; días debe estar entre 0 y 89.")
if (clientes[["compras_90_dias", "dias_ultima_compra"]] % 1 != 0).any().any():
    raise ValueError("Las compras y los días deben ser números enteros.")

print(f"CSV cargado: {len(clientes)} clientes. Datos ficticios para practicar.")
display(clientes.head())


## Celda 2. Preparar los datos y formar grupos
`StandardScaler` estandariza las tres variables para que la magnitud de los pesos no domine
las distancias solo por su unidad. Después, `fit_predict(X)` ajusta **K-Means** y devuelve
el grupo de cada cliente, **sin recibir etiquetas correctas**. [1, 2]

**K = 3** es una decisión de esta práctica, no una cantidad óptima demostrada.
Los números 0, 1 y 2 identifican grupos: no significan malo, regular o bueno.
La **silueta** va de −1 a 1 y resume la separación relativa entre grupos en el espacio
estandarizado: más cerca de 1 indica mayor separación. **No es exactitud ni un porcentaje de aciertos.** [3]

In [ ]:
# CELDA 2. ESTANDARIZAR Y AGRUPAR
K = 3  # Mini reto: cambia este valor a 2 y ejecuta las celdas 2, 3 y 4.
X = clientes[VARIABLES]  # Se excluye cliente_id. No existe una variable objetivo y.
if not isinstance(K, int) or not 2 <= K < len(X) or K > len(X.drop_duplicates()):
    raise ValueError("K debe ser entero: 2 <= K < número de clientes, con suficientes datos distintos.")

escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)
modelo = KMeans(n_clusters=K, random_state=42, n_init=10)
resultado = clientes.copy()
resultado["grupo"] = modelo.fit_predict(X_escalado)

if resultado["grupo"].nunique() != K:
    raise ValueError("No se pudieron formar K grupos distintos. Revisa los datos o reduce K.")
silueta = silhouette_score(X_escalado, resultado["grupo"])
print(f"Grupos formados: {K} | Silueta: {silueta:.3f} (no es un porcentaje)")
display(resultado.head())


## Celda 3. Interpretar los grupos
La tabla muestra el tamaño y los promedios de cada grupo **en las unidades originales**.
El gráfico representa compras y gasto; **el modelo también utilizó los días desde la última compra**,
que no aparecen como un eje. Las X son las proyecciones de los centroides.

**Observa:** ¿qué grupo compra más?, ¿cuál lleva más días sin comprar?
Recuerda: un menor número de días significa una compra más reciente.

In [ ]:
# CELDA 3. PERFIL DE CADA GRUPO Y GRÁFICO
perfiles = resultado.groupby("grupo", as_index=False).agg(
    clientes=("cliente_id", "count"),
    compras_promedio=("compras_90_dias", "mean"),
    gasto_promedio_cop=("gasto_90_dias_cop", "mean"),
    dias_promedio_sin_compra=("dias_ultima_compra", "mean"),
)
display(perfiles.round(1))

# Recuperamos las unidades originales para dibujar los centros correctamente.
centros = escalador.inverse_transform(modelo.cluster_centers_)
figura = plt.figure(figsize=(8, 5))
for grupo, tabla in resultado.groupby("grupo"):
    plt.scatter(tabla["compras_90_dias"], tabla["gasto_90_dias_cop"] / 1000,
                s=65, alpha=0.8, label=f"Grupo {grupo}")
plt.scatter(centros[:, 0], centros[:, 1] / 1000,
            marker="X", s=230, label="Centroides")
for grupo, centro in enumerate(centros):
    plt.annotate(f"G{grupo}", (centro[0], centro[1] / 1000),
                 xytext=(8, 8), textcoords="offset points")
plt.xlabel("Número de compras en 90 días")
plt.ylabel("Gasto en 90 días (miles de pesos colombianos)")
plt.title("Clientes agrupados por su comportamiento de compra")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Celda 4. Proponer campañas y guardar resultados
**Esta parte usa reglas comerciales escritas por nosotros, no aprendizaje automático adicional.**
Como ejemplo, si un grupo promedia al menos 45 días sin comprar, se propone reactivarlo;
si no, y promedia al menos 10 compras, se propone fidelización. Para los demás se propone recompra.
Estos umbrales son **supuestos didácticos modificables**, no conclusiones estadísticas.

Las propuestas se aplican al perfil promedio, no describen necesariamente a todos sus integrantes.
No prueban preferencias de productos, abandono, rentabilidad ni la eficacia de una campaña.
Revísalas y mejora su redacción antes de entregar. No se envía ningún mensaje a clientes.

Se descarga un ZIP con los clientes segmentados, los perfiles con campañas, el gráfico
y un resumen. El ZIP **no incluye el cuaderno**: guarda también tu copia ejecutada.

In [ ]:
# CELDA 4. PROPUESTAS DE MARKETING Y DESCARGA
DIAS_PARA_REACTIVAR = 45  # Supuesto didáctico; no lo aprendió K-Means.
COMPRAS_PARA_FIDELIZAR = 10  # Supuesto didáctico; puedes discutirlo en clase.

def proponer_campana(perfil):
    """Asocia una propuesta humana al perfil promedio del grupo."""
    if perfil["dias_promedio_sin_compra"] >= DIAS_PARA_REACTIVAR:
        return "Reactivación: probar un mensaje de regreso con un incentivo."
    if perfil["compras_promedio"] >= COMPRAS_PARA_FIDELIZAR:
        return "Fidelización: ofrecer beneficios por compras repetidas."
    return "Recompra: probar una oferta para la siguiente compra."

perfiles["campana_sugerida"] = perfiles.apply(proponer_campana, axis=1)
pd.set_option("display.max_colwidth", 100)
display(perfiles[["grupo", "clientes", "campana_sugerida"]])
resultado_final = resultado.merge(
    perfiles[["grupo", "campana_sugerida"]], on="grupo", how="left", validate="many_to_one"
)
resultado_final.to_csv("clientes_segmentados.csv", index=False, encoding="utf-8-sig")
perfiles.round(2).to_csv("perfiles_y_campanas.csv", index=False, encoding="utf-8-sig")
figura.savefig("grafico_segmentacion.png", dpi=150, bbox_inches="tight")

with zipfile.ZipFile("resultados_segmentacion.zip", "w", zipfile.ZIP_DEFLATED) as paquete:
    for archivo in ["clientes_segmentados.csv", "perfiles_y_campanas.csv", "grafico_segmentacion.png"]:
        paquete.write(archivo)
    paquete.writestr("resumen.txt", (
        f"Clientes ficticios: {len(clientes)} | K={K} | Silueta={silueta:.4f}\n"
        f"Variables: {', '.join(VARIABLES)}\n"
        f"Reglas propuestas: reactivar desde {DIAS_PARA_REACTIVAR} días; "
        f"fidelizar desde {COMPRAS_PARA_FIDELIZAR} compras.\n"
        "K es elegido por el estudiante. La silueta no es exactitud.\n"
        "Las campañas son reglas didácticas posteriores, no predicciones de éxito.\n"
        "El modelo se ajustó a todos estos datos para una segmentación exploratoria.\n"
    ))
files.download("resultados_segmentacion.zip")
print("Actividad finalizada. Ing. Freddy Martínez")


## Mini reto y evidencia de aprendizaje
Cambia **K = 3 por K = 2** en la celda 2 y ejecuta de nuevo **2, 3 y 4**.
Compara los resultados con la primera ejecución; no compares números de grupo como si fueran
identidades permanentes. Examina qué clientes quedaron juntos y sus promedios.

Completa aquí esta tabla (doble clic para editar):

| Evidencia | Tu respuesta |
|---|---|
| Silueta con K = 3 y con K = 2 | |
| Un cambio observado en los grupos | |
| Grupo elegido y dos datos que justifican su campaña | |
| Mensaje de campaña propuesto, máximo 25 palabras | |
| ¿Por qué es aprendizaje no supervisado? | |

**Entrega:** cuaderno ejecutado con la tabla respondida y ZIP de resultados de la configuración
que elijas, indicando su K. No necesitas un informe adicional.

**Alcance:** se ajusta el modelo sobre todos los registros porque esta práctica es exploratoria.
No se está evaluando la predicción de ventas ni el rendimiento de una campaña futura.
Una silueta alta con datos sintéticos no demuestra eficacia comercial con clientes reales.
Si la descarga no comienza, busca `resultados_segmentacion.zip` en el panel de archivos del entorno.

## Referencias técnicas
[1. StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) ·
[2. KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) ·
[3. Silhouette score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).

**Relación curricular:** PAD Profundización I - Machine Learning (Conceptos), p. 1, REA 2:
«PROGRAMAR APLICACIONES DE SEGMENTACIÓN Y AGRUPACIÓN EMPLEANDO MODELOS GEOMÉTRICOS
Y PROBABILISTICOS EN SISTEMAS CON GRANDES CONJUNTOS DE DATOS».
Este caso comercial, sus 60 registros y las campañas son una adaptación introductoria creada
para la práctica, no datos ni instrucciones extraídos del plan. No es un ejercicio de gran escala.